In [8]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle

In [9]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [10]:
# with open("fase_3_afrida.pkl", "rb") as f:
#     fase_3_afrida = pickle.load(f)
    
# print("📦 Isi file afrida:")
# for key in fase_3_afrida.keys():
#     print(f" - {key}: {fase_3_afrida[key].shape}")

In [11]:
# The code you provided defines a function `insert_data_to_database` that inserts data into a new database from a dictionary of dataframes. Here's a breakdown of what the function does:
# # Function insert
# def insert_data_to_database(db_connection, cursor, tables_data, tables_to_insert):
#     """
#     Insert data ke database baru dari dictionary dataframes
#     """
#     results = {}
    
#     print("="*80)
#     print("MEMULAI INSERT DATA KE DATABASE BARU")
#     print("="*80)
    
#     for table_name in tables_to_insert:
#         try:
#             if table_name not in tables_data:
#                 print(f"\n⚠️  {table_name}: Tidak ditemukan di data, skip")
#                 results[table_name] = {'status': 'skipped', 'rows': 0}
#                 continue
            
#             df_to_insert = tables_data[table_name]
            
#             if df_to_insert.empty:
#                 print(f"\n⚠️  {table_name}: DataFrame kosong, skip insert")
#                 results[table_name] = {'status': 'empty', 'rows': 0}
#                 continue
            
#             df_to_insert = df_to_insert.dropna(axis=1, how='all')
            
#             columns = ', '.join([f'`{col}`' for col in df_to_insert.columns])
#             placeholders = ', '.join(['%s'] * len(df_to_insert.columns))
            
#             insert_query = f"INSERT INTO `{table_name}` ({columns}) VALUES ({placeholders})"
#             data_to_insert = [tuple(row) for row in df_to_insert.values]
            
#             cursor.executemany(insert_query, data_to_insert)
#             db_connection.commit()
            
#             print(f"✓ {table_name}: Berhasil insert {len(data_to_insert)} baris")
#             results[table_name] = {'status': 'success', 'rows': len(data_to_insert)}
            
#         except Exception as e:
#             db_connection.rollback()
#             print(f"✗ {table_name}: Gagal insert - {e}")
#             results[table_name] = {'status': 'failed', 'error': str(e), 'rows': 0}
    
#     print("\n" + "="*80)
#     print("PROSES INSERT SELESAI")
#     print("="*80)
    
#     return results
    
# # Merge the two dictionaries
# # tables_data = {**fase_2_afrida, **fase_2_cimut}
# tables_to_insert = ['sop_kategori', 'sop',  'surat_keluar', 'verifikasi_surat_keluar', 'surat_tugas', 'surat_tugas_anggota']
# # tables_to_insert += ['users', 'divisions', 'shift_kerja', 'admin_sarpras', 'sop_kategori']
# results = insert_data_to_database(db_new, cursor_new, fase_3_afrida, tables_to_insert)

In [12]:
with open("fase_4_cimut.pkl", "rb") as f:
    fase_4_cimut = pickle.load(f)
    
print("📦 Isi file cimut:")
for key in fase_4_cimut.keys():
    print(f" - {key}: {fase_4_cimut[key].shape}")

📦 Isi file cimut:
 - absensi: (13444, 12)
 - activity_log: (0, 12)
 - admin_sarpras: (1, 2)
 - bidang_kategori: (12, 3)
 - bidang_link: (7, 5)
 - busdev_bidang: (4, 2)
 - cache: (0, 3)
 - cache_locks: (0, 3)
 - calon_siswa: (165, 31)
 - calon_siswa_akademik: (169, 26)
 - calon_siswa_bayar: (169, 8)
 - calon_siswa_jadwal: (169, 9)
 - calon_siswa_kursus: (169, 5)
 - calon_siswa_ortu: (169, 14)
 - calon_siswa_proses: (169, 27)
 - calon_siswa_status_logs: (0, 9)
 - catatan_kelas: (0, 7)
 - catatan_kelas_tag: (0, 3)
 - catatan_mingguan: (0, 7)
 - catatan_siswa: (0, 5)
 - division_user: (0, 5)
 - divisions: (6, 4)
 - failed_jobs: (0, 7)
 - followup_cs: (0, 6)
 - histori_pengajuan: (0, 5)
 - izin_karyawan: (957, 10)
 - jadwal: (0, 9)
 - jadwal_detail: (0, 17)
 - jadwal_detail_logs: (0, 16)
 - jadwal_hari: (0, 3)
 - jadwal_pengajar: (0, 3)
 - jadwal_siswa: (0, 9)
 - job_batches: (0, 10)
 - jobs: (0, 7)
 - kabupaten: (514, 4)
 - karyawan: (51, 36)
 - karyawan_resign: (51, 8)
 - kecamatan: (7266

In [13]:
# Function insert dengan fitur AUTO-SKIP DATA DUPLIKAT & NATIVE LIST EXTRACTION
def insert_data_to_database(db_connection, cursor, tables_data, tables_to_insert):
    """
    Insert data ke database baru dari dictionary dataframes.
    Otomatis skip baris data jika ID (PRIMARY KEY) sudah terdaftar di database.
    """
    results = {}
    
    print("="*80)
    print("MEMULAI INSERT DATA KE DATABASE BARU (AUTO-SKIP DUPLIKAT)")
    print("="*80)
    
    for table_name in tables_to_insert:
        try:
            if table_name not in tables_data:
                print(f"\n⚠️  {table_name}: Tidak ditemukan di data, skip")
                results[table_name] = {'status': 'skipped', 'rows': 0}
                continue
            
            df_to_insert = tables_data[table_name]
            
            if df_to_insert.empty:
                print(f"\n⚠️  {table_name}: DataFrame kosong, skip insert")
                results[table_name] = {'status': 'empty', 'rows': 0}
                continue
            
            df_to_insert = df_to_insert.dropna(axis=1, how='all')
            
            columns = ', '.join([f'`{col}`' for col in df_to_insert.columns])
            placeholders = ', '.join(['%s'] * len(df_to_insert.columns))
            
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns}) VALUES ({placeholders})"
            
            # 💡 PERUBAHAN KRUSIAL: Ubah dataframe menjadi list murni bawaan Python (Native List)
            # Taktik ini otomatis merontokkan error 'int64' dan 'float format code d'
            raw_data_list = df_to_insert.to_numpy().tolist()
            data_to_insert = [tuple(None if pd.isna(x) else x for x in row) for row in raw_data_list]
            
            cursor.executemany(insert_query, data_to_insert)
            db_connection.commit()
            
            rows_affected = cursor.rowcount if cursor.rowcount >= 0 else len(data_to_insert)
            
            print(f"✓ {table_name}: Sukses diproses! Berhasil masuk baru / skip aman sebanyak {len(data_to_insert)} baris data.")
            results[table_name] = {'status': 'success', 'rows': rows_affected}
            
        except Exception as e:
            db_connection.rollback()
            print(f"✗ {table_name}: Gagal insert - {e}")
            results[table_name] = {'status': 'failed', 'error': str(e), 'rows': 0}
    
    print("\n" + "="*80)
    print("PROSES INSERT SELESAI DENGAN AMAN")
    print("="*80)
    
    return results

# === JALANKAN ULANG SEKARANG JUGA! ===
tables_to_insert = ['izin_karyawan',
    'absensi',
    'verifikasi_absensi',
    'verifikasi_izin',
    'karyawan_resign']
results = insert_data_to_database(db_new, cursor_new, fase_4_cimut, tables_to_insert)

MEMULAI INSERT DATA KE DATABASE BARU (AUTO-SKIP DUPLIKAT)
✓ izin_karyawan: Sukses diproses! Berhasil masuk baru / skip aman sebanyak 957 baris data.
✓ absensi: Sukses diproses! Berhasil masuk baru / skip aman sebanyak 13444 baris data.
✓ verifikasi_absensi: Sukses diproses! Berhasil masuk baru / skip aman sebanyak 11 baris data.
✓ verifikasi_izin: Sukses diproses! Berhasil masuk baru / skip aman sebanyak 2107 baris data.
✓ karyawan_resign: Sukses diproses! Berhasil masuk baru / skip aman sebanyak 51 baris data.

PROSES INSERT SELESAI DENGAN AMAN


In [14]:
# # Truncate tables yang sudah diinsert (dengan force delete)
# tables_to_truncate = tables_to_insert

# print("="*80)
# print("MEMULAI TRUNCATE DATA DI DATABASE BARU")
# print("="*80)

# # Disable foreign key checks
# cursor_new.execute("SET FOREIGN_KEY_CHECKS=0")
# db_new.commit()

# for sop in tables_to_truncate:
#     try:
#         truncate_query = f"TRUNCATE TABLE `{sop}`"
#         cursor_new.execute(truncate_query)
#         db_new.commit()
#         print(f"✓ {sop}: Berhasil truncate")
#     except Exception as e:
#         db_new.rollback()
#         print(f"✗ {sop}: Gagal truncate - {e}")

# # Re-enable foreign key checks
# cursor_new.execute("SET FOREIGN_KEY_CHECKS=1")
# db_new.commit()

# print("\n" + "="*80)
# print("PROSES TRUNCATE SELESAI")
# print("="*80)